In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

imagen_bgr = cv2.imread('colores.jpg')
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

ancho = imagen_rgb.shape[1]
alto  = imagen_rgb.shape[0]
paso  = ancho // 5
margen = 30

columnas = {
    "Red":    imagen_rgb[:, 0*paso : 1*paso - margen],
    "Orange": imagen_rgb[:, 1*paso : 2*paso - margen],
    "Yellow": imagen_rgb[:, 2*paso : 3*paso - margen],
    "Green":  imagen_rgb[:, 3*paso : 4*paso - margen],
    "Blue":   imagen_rgb[:, 4*paso : ancho],
}

config = {
    "Red":    {"rangos": [(np.array([0,   60, 50]), np.array([9,   255, 255])),
                          (np.array([170, 60, 50]), np.array([180, 255, 255]))], "min_area": 80},
    "Orange": {"rangos": [(np.array([9,  60, 50]),  np.array([22, 255, 255]))],  "min_area": 80},
    "Yellow": {"rangos": [(np.array([22, 30, 50]),  np.array([38, 255, 255]))],  "min_area": 25},
    "Green":  {"rangos": [(np.array([35, 25, 25]),  np.array([95, 255, 255]))],  "min_area": 20},
    "Blue":   {"rangos": [(np.array([85, 35, 35]),  np.array([140, 255, 255]))], "min_area": 20},
}

kernel = np.ones((3, 3), np.uint8)
conteos = {}

for color, col_rgb in columnas.items():
    col_hsv = cv2.cvtColor(col_rgb, cv2.COLOR_RGB2HSV)
    mask = np.zeros(col_hsv.shape[:2], dtype=np.uint8)
    for lower, upper in config[color]["rangos"]:
        mask = cv2.bitwise_or(mask, cv2.inRange(col_hsv, lower, upper))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    count = 0
    for lbl in range(1, num_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= config[color]["min_area"]:
            count += 1
    conteos[color] = count

plt.figure(figsize=(15, 8))

plt.subplot(231)
plt.imshow(imagen_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(232)
plt.imshow(columnas["Red"])
plt.title(f'Red objects:{conteos["Red"]}')
plt.axis('off')

plt.subplot(233)
plt.imshow(columnas["Orange"])
plt.title(f'Orange objects:{conteos["Orange"]}')
plt.axis('off')

plt.subplot(234)
plt.imshow(columnas["Yellow"])
plt.title(f'Yellow objects:{conteos["Yellow"]}')
plt.axis('off')

plt.subplot(235)
plt.imshow(columnas["Green"])
plt.title(f'Green objects:{conteos["Green"]}')
plt.axis('off')

plt.subplot(236)
plt.imshow(columnas["Blue"])
plt.title(f'Blue objects:{conteos["Blue"]}')
plt.axis('off')

plt.tight_layout()
plt.show()

print("Conteo final:")
for color in ["Red", "Orange", "Yellow", "Green", "Blue"]:
    print(f"  {color}: {conteos[color]}")

error: OpenCV(4.12.0) C:\Users\dev-admin\buildout\croot\opencv-suite_1765297159650\work\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
